# TB cross-domain judge run

Run the cells in order. Groq is free and comes first; OpenAI costs about
$0.008 and comes second. Each judge has its own cell, so you can stop after
Groq if you want.

When you are done, copy everything printed by the last cell and send it.


## 1. Install

In [ ]:
!pip -q install openai

## 2. Keys

Set either or both. Groq: console.groq.com (free, no card).
OpenAI: platform.openai.com (prepaid, $5 minimum).


In [ ]:
GROQ_KEY   = ""   # gsk_...   free
OPENAI_KEY = ""   # sk-...    ~$0.008

assert GROQ_KEY or OPENAI_KEY, "Set at least one key."
print("Groq:  ", "set" if GROQ_KEY else "not set")
print("OpenAI:", "set" if OPENAI_KEY else "not set")

## 3. Build the probe

Pulls WHO's published report data and constructs the 38 items. No API calls.

In [ ]:
import csv, io, json, time, urllib.request
from math import comb, sqrt

# ---------- 1. build the probe from WHO's own report data ----------
CSV_URL = ("https://raw.githubusercontent.com/GTB-TME/gtbreport2024/"
           "main/data/ps/prev_notif_2023-07-03.csv")

STOCK_T = ("A national TB prevalence survey in {country} estimated "
           "{v} prevalent cases of TB in {year}.")
FLOW_T  = ("{v} TB cases were notified to national authorities in "
           "{country} during {year}.")
Q_STOCK = ("How many prevalent cases of TB were estimated by the national "
           "prevalence survey in {country} in {year}?")
Q_FLOW  = ("How many TB cases were notified to national authorities in "
           "{country} during {year}?")

fmt   = lambda n: f"{round(n):,}"
delta = lambda a, b: abs(b - a) / a if a else float("inf")

raw = urllib.request.urlopen(CSV_URL).read().decode("utf-8")
recs = []
for r in csv.DictReader(io.StringIO(raw)):
    try:
        pr, pop, nt = float(r["prev"]), float(r["pop"]), float(r["notified"])
    except (ValueError, KeyError):
        continue
    stock = pr / 1e5 * pop
    if stock > 0 and nt > 0:
        recs.append({"country": r["country"].split("(")[0].strip(),
                     "year": r["year"], "stock": stock, "flow": nt})

probe = []
for rec in recs:
    others = [o for o in recs if o is not rec]
    for gt in ("STOCK", "FLOW"):
        gv = rec[gt.lower()]
        nv = rec["flow" if gt == "STOCK" else "stock"]
        dn = delta(gv, nv)
        if not (0.03 <= dn <= 1.0):
            continue
        cands = []
        for o in others:
            ov = o["flow" if gt == "STOCK" else "stock"]
            d = delta(gv, ov)
            if 5.0 <= d <= 500.0:
                cands.append((d, o, ov))
        if not cands:
            continue
        cands.sort(key=lambda x: x[0])
        df, orec, ov = cands[len(cands) // 2]

        g_t = STOCK_T if gt == "STOCK" else FLOW_T
        n_t = FLOW_T  if gt == "STOCK" else STOCK_T
        gold = g_t.format(v=fmt(gv), country=rec["country"], year=rec["year"])
        near = n_t.format(v=fmt(nv), country=rec["country"], year=rec["year"])
        far  = n_t.format(v=fmt(ov), country=orec["country"], year=orec["year"])

        probe.append({
            "id": f"tb{len(probe)+1:02d}",
            "question": (Q_STOCK if gt == "STOCK" else Q_FLOW).format(
                country=rec["country"], year=rec["year"]),
            "evidence": " ".join([gold, near, far]),
            "answers": {"correct": gold,
                        "ATT_near": gold.replace(fmt(gv), fmt(nv), 1),
                        "ATT_far":  gold.replace(fmt(gv), fmt(ov), 1),
                        "SUB_near": near,
                        "SUB_far":  far},
        })

print(f"probe built: {len(probe)} items")
json.dump(probe, open("probe_tb.json", "w"), indent=1)

RESULTS = {}

## 4. Judge helpers

The two prompts and the run loop. No API calls yet.

In [ ]:
FAITH = ("Evaluate FAITHFULNESS: are the ANSWER's claims supported by the "
         "EVIDENCE?\nReply with only 1 (supported) or 0 (not supported).\n\n"
         "EVIDENCE:\n{evidence}\n\nANSWER:\n{answer}\n")
REL   = ("Evaluate ANSWER_RELEVANCY: does the ANSWER address the QUESTION "
         "asked?\nReply with only 1 (addresses it) or 0 (does not).\n\n"
         "QUESTION:\n{question}\n\nANSWER:\n{answer}\n")
ROWS = ["correct", "ATT_near", "ATT_far", "SUB_near", "SUB_far"]

from openai import OpenAI

def ask(cl, model, prompt, tries=4):
    for a in range(tries):
        try:
            r = cl.chat.completions.create(model=model, temperature=0,
                                           max_tokens=8,
                                           messages=[{"role": "user",
                                                      "content": prompt}])
            t = (r.choices[0].message.content or "").strip()
            if t.startswith("1"): return 1.0
            if t.startswith("0"): return 0.0
            if "1" in t and "0" not in t: return 1.0
            if "0" in t and "1" not in t: return 0.0
            return None
        except Exception as e:
            print(f"    retry {a+1}: {str(e)[:70]}")
            time.sleep(15 * (a + 1))
    return None

def run(model, key, base=None, out=None):
    cl = OpenAI(api_key=key, **({"base_url": base} if base else {}))
    res, drop = [], 0
    for i, it in enumerate(probe, 1):
        for row in ROWS:
            ans = it["answers"][row]
            f = ask(cl, model, FAITH.format(evidence=it["evidence"], answer=ans))
            v = ask(cl, model, REL.format(question=it["question"], answer=ans))
            if f is None or v is None:
                drop += 1
                continue
            res.append({"id": it["id"], "row": row, "faith": f, "rel": v})
        if i % 5 == 0:
            print(f"  {model}: {i}/{len(probe)} items")
    with open(out, "w") as fh:
        for r in res:
            fh.write(json.dumps(r) + "\n")
    print(f"  {model}: {len(res)} verdicts, {drop} unparseable -> {out}")
    return res

## 5. Groq judge — FREE

`gpt-oss-20b`, the replication judge. Costs nothing. Run this first: if it
finishes, the pipeline works.


In [ ]:
if GROQ_KEY:
    print("running gpt-oss-20b on Groq (free) ...")
    RESULTS["openai/gpt-oss-20b (replication)"] = run(
        "openai/gpt-oss-20b", GROQ_KEY,
        "https://api.groq.com/openai/v1", "judge_gpt-oss-20b.jsonl")
    print("done - Groq judge finished")
else:
    print("no GROQ_KEY set, skipping")

## 6. OpenAI judge — PAID (~$0.008)

`gpt-4o-mini`, the primary judge. This is the one that matches the
displacement table in the paper. Skip this cell if you only want the free run.


In [ ]:
if OPENAI_KEY:
    print("running gpt-4o-mini on OpenAI (paid, ~$0.008) ...")
    RESULTS["gpt-4o-mini (primary)"] = run(
        "gpt-4o-mini", OPENAI_KEY, None, "judge_gpt-4o-mini.jsonl")
    print("done - OpenAI judge finished")
else:
    print("no OPENAI_KEY set, skipping")

## 7. Analysis

Prints a block per judge, plus the cross-domain table and a LaTeX body.

In [ ]:
# ---------- 3. analysis ----------
Z = 1.959964
def wilson(k, n):
    if not n: return (0.0, 0.0)
    p = k / n; d = 1 + Z*Z/n
    c = (p + Z*Z/(2*n)) / d
    h = Z * sqrt(p*(1-p)/n + Z*Z/(4*n*n)) / d
    return max(0, c-h)*100, min(1, c+h)*100

def auc(p1, p2):
    return p1*(1-p2) + 0.5*(p1*p2 + (1-p1)*(1-p2))

DISP = {"correct": (57, 55), "ATT_near": (0, 55), "ATT_far": (0, 11),
        "SUB_near": (57, 36), "SUB_far": (57, 3)}

for name, res in RESULTS.items():
    agg = {r: {"faith": [0, 0], "rel": [0, 0]} for r in ROWS}
    for r in res:
        for c in ("faith", "rel"):
            agg[r["row"]][c][1] += 1
            agg[r["row"]][c][0] += int(r[c] == 1.0)
    print(f"\n===== TB — {name} =====")
    print(f"{'class':10s} {'faithful':>14s} {'relevant':>14s}")
    for r in ROWS:
        fk, fn = agg[r]["faith"]; rk, rn = agg[r]["rel"]
        print(f"{r:10s} {fk:3d}/{fn:<3d}={fk/fn if fn else 0:.3f} "
              f"{rk:3d}/{rn:<3d}={rk/rn if rn else 0:.3f}")
    print("\nWilson 95% CI on faithfulness:")
    for r in ROWS:
        fk, fn = agg[r]["faith"]; lo, hi = wilson(fk, fn)
        print(f"  {r:10s} {100*fk/fn if fn else 0:5.1f}%  [{lo:.1f}, {hi:.1f}]")
    ck, cn = agg["correct"]["rel"]; pc = ck/cn if cn else 0
    print("\nRelevance AUC vs correct:")
    for r in ROWS[1:]:
        rk, rn = agg[r]["rel"]
        print(f"  {r:10s} {auc(pc, rk/rn if rn else 0):.3f}")
    print("\nLaTeX table body:")
    lbl = {"correct": "correct & ---", "ATT_near": r"\attS          & near",
           "ATT_far": r"\attS          & far ", "SUB_near": r"\subS          & near",
           "SUB_far": r"\subS          & far "}
    for r in ROWS:
        fk, fn = agg[r]["faith"]; rk, rn = agg[r]["rel"]
        print(f"  {lbl[r]} & {fk/fn if fn else 0:.3f} & {rk/rn if rn else 0:.3f} \\\\")
    if True:
        print(f"\n===== CROSS-DOMAIN  (TB column = {name}) =====")
        print(f"{'class':10s} {'displacement':>14s} {'TB':>14s}")
        for r in ROWS:
            dk = DISP[r][0]; fk, fn = agg[r]["faith"]
            print(f"{r:10s} {dk:3d}/57 ={dk/57:.3f} {fk:3d}/{fn:<3d}="
                  f"{fk/fn if fn else 0:.3f}")

print("\nDone. Copy everything above and send it.")